# Preparing data for TTS transformer

### Tokenizing text

In [1]:
import re
from itertools import product
from typing import List, Tuple

import IPython
import numpy as np
import pandas as pd
import torch
import torchaudio
import umap
from datasets import Audio, load_dataset
from phonemizer import phonemize
from phonemizer.separator import Separator
from plotly import express as px
from sklearn_extra.cluster import KMedoids
from transformers import AutoProcessor, EncodecModel

In [2]:
def extract_phonemes(text: str) -> str:
    """Convertes a text into phonemes.
    Words are separated with ' | ' and phonemes with '#'.

    Args:
        text (str): original text

    Returns:
        str: phoneme representation of a text
    """
    phonemes = phonemize(
        text,
        language="ru",
        backend="espeak",
        separator=Separator(phone="#", word=" | "),
        preserve_punctuation=False,
        with_stress=True,
        strip=True,
    )
    return phonemes

In [3]:
# preprocessing original text corpus
pattern = re.compile(r"\\u[0-9a-fA-F]{4}\d?")
corpus = ""
with open("../data/corpus.txt", encoding="utf-8") as f:
    for line in f.readlines():
        line = re.sub(r"[\u2020-\u203f]\d?", "", line)
        corpus += line

In [4]:
# creating vocab of phonemes
text_vocab = [
    "<s_ph>",
    "</s_ph>",
    "</s_mel>",
    "<s_mel>",
    "<UNK_ph>",
    "<UNK_mel>",
    "<space>",
]
for word in extract_phonemes(corpus).split(" | "):
    for phoneme in word.split("#"):
        if phoneme.isalpha() and phoneme not in text_vocab:
            text_vocab.append(phoneme)
text_vocab = sorted(text_vocab)

In [5]:
len(text_vocab)

69

In [6]:
phon_to_idx = {phoneme: i for i, phoneme in enumerate(text_vocab)}
idx_to_phon = {i: phoneme for i, phoneme in enumerate(text_vocab)}


def tokenize_text(text: str) -> List[int]:
    """Converts text into phonemes and tokenizes it.

    Args:
        text (str): original text

    Returns:
        List[int]:tokenized representation of a text
    """
    phonemes = extract_phonemes(text)
    tokens = [phon_to_idx["<s_ph>"]]
    for word in phonemes.split(" | "):
        for phoneme in word.split("#"):
            if phoneme in phon_to_idx.keys():
                tokens.append(phon_to_idx[phoneme])
            else:
                tokens.append(phon_to_idx["<UNK_ph>"])
        tokens.append(phon_to_idx["<space>"])
    tokens.append(phon_to_idx["</s_ph>"])
    return tokens


def untokenize_text(tokens: List[int]) -> List[str]:
    """Converts text tokens back into phonemes

    Args:
        tokens (List[int]): tokenized representation of text

    Returns:
        List[str]: list of phonemes separated by service tokens
    """
    phonemes = []
    for token in tokens:
        if token in idx_to_phon.keys():
            phonemes.append(idx_to_phon[token])
        else:
            phonemes.append("<UNK_ph>")
    return phonemes

In [7]:
text = "Это обычное тестовое предложение."
tokens = tokenize_text(text)
new_text = untokenize_text(tokens)
print(tokens)
print(new_text)

[5, 64, 33, 53, 6, 53, 8, 62, 35, 24, 53, 17, 49, 6, 36, 64, 31, 33, 53, 38, 53, 17, 49, 6, 27, 30, 16, 10, 50, 53, 55, 64, 25, 16, 17, 49, 6, 1]
['<s_ph>', 'ˈɛ', 't', 'ʌ', '<space>', 'ʌ', 'b', 'ˈy', 'tʃʲ', 'n', 'ʌ', 'j', 'ɪ', '<space>', 'tʲ', 'ˈɛ', 's', 't', 'ʌ', 'v', 'ʌ', 'j', 'ɪ', '<space>', 'p', 'rʲ', 'i', 'd', 'ɭ', 'ʌ', 'ʒ', 'ˈɛ', 'nʲ', 'i', 'j', 'ɪ', '<space>', '</s_ph>']


### Tokenizing audio

In [8]:
# testing encodec model on dummy audio

librispeech_dummy = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy", "clean", split="validation"
)
model = EncodecModel.from_pretrained("facebook/encodec_24khz")
processor = AutoProcessor.from_pretrained("facebook/encodec_24khz")
librispeech_dummy = librispeech_dummy.cast_column(
    "audio", Audio(sampling_rate=processor.sampling_rate)
)
audio_sample = librispeech_dummy[-1]["audio"]["array"]

inputs = processor(
    raw_audio=audio_sample, sampling_rate=processor.sampling_rate, return_tensors="pt"
)
with torch.no_grad():
    encoder_outputs = model.encode(inputs["input_values"], inputs["padding_mask"])
    audio_values = model.decode(
        encoder_outputs.audio_codes,
        encoder_outputs.audio_scales,
        inputs["padding_mask"],
    )

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [9]:
orig_audio = torch.tensor(audio_sample).unsqueeze(dim=0)
torchaudio.save("../artifacts/orig_audio.mp3", orig_audio, sample_rate=24000)
print("Original audio:")
IPython.display.display(IPython.display.Audio("../artifacts/orig_audio.mp3"))

encodec_audio = audio_values.audio_values.squeeze(0)
torchaudio.save("../artifacts/encodec_audio.mp3", encodec_audio, sample_rate=24000)
print("Audio processed by encodec:")
IPython.display.Audio("../artifacts/encodec_audio.mp3")

Original audio:


Audio processed by encodec:


In [10]:
# exploring codebooks representations
second_codebook = model.quantizer.layers[1].codebook
embeddings = second_codebook.embed.numpy()
print("Embeddings shape:", embeddings.shape)

# clusterizing embeddings in a second codebook to reduce number of future audio tokens
clusterized = KMedoids(n_clusters=78, random_state=42).fit(embeddings)
labels = clusterized.labels_
centroids = clusterized.cluster_centers_
print("Centroids shape:", centroids.shape)
combined_embeddings = np.vstack([embeddings, centroids])
print("Resulting shape:", combined_embeddings.shape)

# preparing data for clusterization visualization
reducer = umap.UMAP(n_components=3)
embeddings_3d = reducer.fit_transform(combined_embeddings)
print("Reduced shape:", embeddings_3d.shape)

Embeddings shape: (1024, 128)
Centroids shape: (78, 128)
Resulting shape: (1102, 128)
Reduced shape: (1102, 3)


In [11]:
df = pd.DataFrame.from_dict(
    [
        {
            "x": embeddings_3d[i, 0],
            "y": embeddings_3d[i, 1],
            "z": embeddings_3d[i, 2],
            "label": str(labels[i]),
            "symbol": "circle",
            "size_col": 4,
        }
        for i in range(1024)
    ]
    + [
        {
            "x": embeddings_3d[1024 + i, 0],
            "y": embeddings_3d[1024 + i, 1],
            "z": embeddings_3d[1024 + i, 2],
            "label": str(i),
            "symbol": "star",
            "size_col": 10,
        }
        for i in range(78)
    ]
)

In [12]:
df.head(3)

,x,y,z,label,symbol,size_col
0,4.021497,7.014215,2.651751,66,circle,4
1,5.384695,7.498250,1.637500,32,circle,4
2,4.653236,5.781970,0.936362,44,circle,4


In [13]:
df.tail(3)

,x,y,z,label,symbol,size_col
1099,5.629758,5.164687,0.783879,75,star,10
1100,4.208596,5.748016,1.879951,76,star,10
1101,2.599019,7.170362,4.171669,77,star,10


In [14]:
fig = px.scatter_3d(
    df,
    x="x",
    y="y",
    z="z",
    color="label",
    size="size_col",
    symbol="symbol",
    width=1000,
    height=700,
)
fig.update_traces(
    marker=dict(opacity=1, line=dict(width=0, color="DarkSlateGrey")),
    selector=dict(mode="markers"),
)
fig.update_layout(
    title="<b>3D-projection of 2nd codebook embeddings clusters</b>",
)
fig.show()

In [15]:
# dict to convert second codebooks quantized embeddings to its clusters centroids
second_cb_to_ind = {embedding: label for embedding, label in zip(range(1024), labels)}
# dict to convert labels of second codebook to second codebook quantized embeddings
ind_to_second_cb = {label: centroid for label, centroid in enumerate(centroids)}


# dicts to convert quantized embeddings of encodec model to audio tokens and back
text_vocab_size = len(text_vocab)
encodec_to_tokens = {
    tuple(encodec_codes): token
    + text_vocab_size  # adding text vocab size to prevent overlapping between text and audio tokens
    for token, encodec_codes in enumerate(list(product(range(1024), range(78))))
}
tokens_to_encodec = {
    token + text_vocab_size: tuple(encodec_codes)
    for token, encodec_codes in enumerate(list(product(range(1024), range(78))))
}

In [19]:
def process_audio(path_to_audio: str) -> np.ndarray[float]:
    """Loads audio from path and converts it to numpy array.

    Args:
        path_to_audio (str): location of audio file

    Returns:
        np.ndarray: waveform representation of audio sample
    """
    waveform, orig_sr = torchaudio.load(path_to_audio)
    resampler = torchaudio.transforms.Resample(orig_freq=orig_sr, new_freq=24000)
    waveform = resampler(waveform)
    return waveform


def tokenize_waveform(waveform: np.ndarray[float]) -> Tuple[List[int], torch.tensor]:
    """Converts waveform representation of audio sample into tokens.

    Args:
        waveform (np.ndarray): waveform representation of audio

    Returns:
        Tuple[List[int], torch.tensor[[int]]]: tokenized audio and padding mask used in encodec model
    """
    model = EncodecModel.from_pretrained("facebook/encodec_24khz")
    processor = AutoProcessor.from_pretrained("facebook/encodec_24khz")

    inputs = processor(
        raw_audio=waveform, sampling_rate=processor.sampling_rate, return_tensors="pt"
    )
    with torch.no_grad():
        encoder_outputs = model.encode(
            inputs["input_values"], inputs["padding_mask"]
        ).audio_codes.squeeze()

    first_cb_codes = encoder_outputs[0]
    second_cb_codes = encoder_outputs[1]
    new_second_codes = torch.tensor(
        [second_cb_to_ind[int(code)] for code in second_cb_codes]
    )
    new_codes = torch.vstack([first_cb_codes, new_second_codes])

    tokenized = []
    for encoding in new_codes.transpose(1, 0):
        encoding = tuple([int(encoding[0]), int(encoding[1])])
        tokenized.append(encodec_to_tokens[encoding])
    return tokenized, inputs["padding_mask"]


def tokens_to_audio(audio_tokens: List[int], padding_mask: torch.tensor):
    """Converts tokens to audio representation

    Args:
        audio_tokens (List[int]): tokenized representation of original audio
        padding_mask (torch.tensor[[int]]): padding mask used in encodec model

    Returns:
        EncodecDecoderOutput: output audio representation of encodec model
    """
    model = EncodecModel.from_pretrained("facebook/encodec_24khz")
    encodec_tokens = []
    for token in audio_tokens:
        encodec_tokens.append(tokens_to_encodec[token])
    encodec_tokens = (
        torch.tensor(encodec_tokens).transpose(1, 0).unsqueeze(dim=0).unsqueeze(dim=0)
    )
    with torch.no_grad():
        audio_values = model.decode(
            encodec_tokens,
            [None],
            padding_mask,
        )
    return audio_values

In [20]:
tokens, padding_mask = tokenize_waveform(audio_sample)
print(tokens)
print(padding_mask)

[65243, 65243, 65212, 65204, 65233, 65204, 65229, 65229, 65204, 65229, 65204, 65229, 31898, 65204, 57646, 31912, 31912, 31898, 31912, 31912, 31912, 57646, 31912, 31912, 31912, 31898, 31912, 31898, 31898, 31927, 57646, 31912, 31898, 31900, 31898, 31912, 31912, 31898, 31912, 31912, 26541, 65165, 30923, 68340, 74780, 52350, 63404, 52373, 18620, 4245, 39947, 26867, 14699, 62911, 41921, 14699, 11823, 187, 59918, 59043, 44530, 77367, 74593, 44530, 2863, 41900, 2978, 14304, 52355, 47214, 60902, 68410, 33627, 66725, 39035, 39031, 48998, 20918, 57514, 69849, 56511, 15973, 57369, 45717, 38223, 56745, 19809, 49101, 29207, 8147, 26569, 36207, 8147, 3092, 39951, 68381, 48934, 47321, 77406, 77371, 52341, 17870, 65219, 57638, 37133, 30913, 65009, 39952, 73043, 62357, 62318, 36071, 73011, 36563, 59629, 69907, 63830, 4521, 19317, 57105, 4266, 4261, 36206, 36206, 4226, 36227, 29207, 36227, 36213, 26578, 8377, 57663, 65218, 36227, 4247, 4247, 71093, 581, 54245, 76787, 7212, 23703, 49156, 68421, 36223, 42

In [21]:
audio = tokens_to_audio(tokens, padding_mask)
print(audio)

EncodecDecoderOutput(audio_values=tensor([[[ 0.0002, -0.0013, -0.0007,  ...,  0.0032,  0.0030,  0.0031]]]))


In [22]:
waveform = audio.audio_values.squeeze(0)
torchaudio.save("../artifacts/decoded_audio.mp3", waveform, sample_rate=24000)
print("Audio processed by encodec with clusterized 2nd codebook:")
IPython.display.display(IPython.display.Audio("../artifacts/decoded_audio.mp3"))

Audio processed by encodec with clusterized 2nd codebook:
